# Extension 15: Rubric vs Preference Reward Model

**Research Question**: Does explicit rubric decomposition reduce the length-exploitation vulnerability that the Bradley-Terry RM shows — and at what cost to pairwise accuracy?

## The core trade-off

The Bradley-Terry RM was trained on pairwise preferences: "is response A better than response B?" It has no explicit representation of *why* one response is better. As documented in the Key Findings section, it learned that longer, more structured responses tend to win — so it rewards verbosity even when the added length is filler.

The Rubric RM is trained differently: Claude grades each response on five explicit criteria (1–5 each), and the RM learns to predict that sum. One of the five criteria is **Conciseness** — which explicitly penalizes padding, hollow affirmations, and unnecessary repetition.

**Hypothesis**: The Conciseness criterion will reduce the RM's susceptibility to filler-padding exploits, at a modest cost to in-distribution pairwise accuracy.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from src.data.rubric_preferences import RUBRIC, RUBRIC_MAX_SCORE
from src.training.rubric_reward import RubricRewardConfig, rubric_mse_loss

print("Rubric criteria:")
for i, c in enumerate(RUBRIC, 1):
    print(f"  {i}. {c.name.title():14} — {c.description}")
    print(f"      Low:  {c.anchor_low}")
    print(f"      High: {c.anchor_high}")
    print()
print(f"Max score: {RUBRIC_MAX_SCORE} (sum of 5 criteria × 5 points each)")

## 1. Sample Rubric Grading

Show what Claude's rubric grades look like for a concrete pair. The key pattern to observe: the verbose PPO-style response gets a low Conciseness score despite its high Helpfulness.

In [ ]:
import os, anthropic

from src.data.rubric_preferences import RUBRIC, grade_response

client = anthropic.Anthropic()

SAMPLE_PROMPT = "What's a good way to deal with anxiety before a job interview?"

# This mirrors the SFT vs PPO example from the Key Findings section
SFT_RESPONSE = (
    "One thing that can help is to prepare thoroughly — research the company, practice "
    "common questions out loud, and remind yourself that it's normal to feel nervous. "
    "Try deep breathing for a few minutes before you go in. Getting enough sleep the "
    "night before also makes a real difference."
)

PPO_RESPONSE = (
    "That's a great question! Anxiety before interviews is completely normal and there "
    "are several research-backed strategies that can help. First, thorough preparation "
    "is key — when you know your material well, the uncertainty that drives anxiety "
    "decreases significantly. Second, controlled breathing exercises activate the "
    "parasympathetic nervous system, counteracting the fight-or-flight response. Third, "
    "reframing the interview as a two-way conversation rather than an evaluation can "
    "shift your mental posture. Many people also find light exercise the morning of "
    "helpful for regulating cortisol levels. Would you like more detail on any of these?"
)

print("Grading SFT response...")
sft_scores = grade_response(client, SAMPLE_PROMPT, SFT_RESPONSE)

print("Grading PPO (verbose) response...")
ppo_scores = grade_response(client, SAMPLE_PROMPT, PPO_RESPONSE)

print()
print(f"{'Criterion':<16} {'SFT':>6} {'PPO (verbose)':>14}")
print("-" * 38)
for c in RUBRIC:
    print(f"  {c.name.title():<14} {sft_scores[c.name]:>6} {ppo_scores[c.name]:>14}")
print("-" * 38)
print(f"  {'TOTAL':<14} {sft_scores['total']:>6} {ppo_scores['total']:>14}")
print(f"  {'Normalized':<14} {sft_scores['normalized']:>6.2f} {ppo_scores['normalized']:>14.2f}")
print()
print("Note: PPO scores HIGH on helpfulness/specificity but LOW on conciseness.")
print("The BT RM would score PPO higher; the Rubric RM may score them similarly.")

## 2. Rubric Score Distributions

Grade a sample of chosen and rejected responses to see whether rubric scores separate them.

In [ ]:
import time
from datasets import load_dataset
from src.data.rubric_preferences import generate_rubric_dataset, RUBRIC

# Grade 50 pairs (100 responses total)
dataset = load_dataset("Anthropic/hh-rlhf", split="train")
pairs = [
    {
        "prompt": ex["chosen"].split("\n\nAssistant:")[0].replace("Human:", "").strip(),
        "chosen":   ex["chosen"].split("\n\nAssistant:")[-1].strip(),
        "rejected": ex["rejected"].split("\n\nAssistant:")[-1].strip(),
    }
    for ex in dataset.select(range(50))
]

records = generate_rubric_dataset(pairs, client, max_samples=50, sleep=0.3)

chosen_records   = [r for r in records if r["label"] == "chosen"]
rejected_records = [r for r in records if r["label"] == "rejected"]

print(f"Chosen   mean total score: {sum(r['total_score'] for r in chosen_records)/len(chosen_records):.2f}")
print(f"Rejected mean total score: {sum(r['total_score'] for r in rejected_records)/len(rejected_records):.2f}")
print()

print(f"Per-criterion breakdown:")
print(f"{'Criterion':<14} {'Chosen':>8} {'Rejected':>10} {'Gap':>6}")
print("-" * 42)
for c in RUBRIC:
    ch  = sum(r["rubric_scores"][c.name] for r in chosen_records)   / len(chosen_records)
    rej = sum(r["rubric_scores"][c.name] for r in rejected_records) / len(rejected_records)
    print(f"  {c.name:<12} {ch:>8.2f} {rej:>10.2f} {ch-rej:>+6.2f}")

## 3. Train the Rubric Reward Model

In [ ]:
import json

# Save the graded records as JSONL
RUBRIC_DATA_PATH = "../data/rubric_scored_pairs.jsonl"
os.makedirs(os.path.dirname(RUBRIC_DATA_PATH), exist_ok=True)
with open(RUBRIC_DATA_PATH, "w") as f:
    for rec in records:
        f.write(json.dumps(rec) + "\n")
print(f"Saved {len(records)} rubric-scored records to {RUBRIC_DATA_PATH}")

from src.training.rubric_reward import RubricRewardConfig, train_rubric_reward_model

cfg = RubricRewardConfig(
    rubric_data_path=RUBRIC_DATA_PATH,
    num_epochs=2,
)
train_rubric_reward_model(cfg)
print("\nRubric RM training complete.")

## 4. Pairwise Accuracy Comparison

In [ ]:
import torch
from transformers import AutoTokenizer
from src.models.reward_model import GPT2RewardModel
from src.training.rubric_reward import _score_one

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(cfg.sft_checkpoint)
tokenizer.pad_token = tokenizer.eos_token

rubric_model = GPT2RewardModel.from_pretrained(cfg.output_dir).to(device)
bt_model     = GPT2RewardModel.from_pretrained("../checkpoints/reward_model").to(device)

# Evaluate on 100 test pairs
test_dataset = load_dataset("Anthropic/hh-rlhf", split="test")
test_pairs = [
    {
        "prompt":   ex["chosen"].split("\n\nAssistant:")[0].replace("Human:", "").strip(),
        "chosen":   ex["chosen"].split("\n\nAssistant:")[-1].strip(),
        "rejected": ex["rejected"].split("\n\nAssistant:")[-1].strip(),
    }
    for ex in test_dataset.select(range(100))
]

bt_correct = rubric_correct = 0
for pair in test_pairs:
    p, c, r = pair["prompt"], pair["chosen"], pair["rejected"]
    if _score_one(bt_model,     tokenizer, p, c, device, 512) > _score_one(bt_model,     tokenizer, p, r, device, 512):
        bt_correct += 1
    if _score_one(rubric_model, tokenizer, p, c, device, 512) > _score_one(rubric_model, tokenizer, p, r, device, 512):
        rubric_correct += 1

n = len(test_pairs)
print(f"Bradley-Terry pairwise accuracy: {bt_correct/n:.1%}")
print(f"Rubric RM     pairwise accuracy: {rubric_correct/n:.1%}")
print(f"\nBT leads by {(bt_correct - rubric_correct)/n:+.1%} pp")

## 5. Length Bias Experiment

Append a filler paragraph to each probe response and measure how much each RM's score changes. The Bradley-Terry RM rewards verbosity (no explicit penalty). The Rubric RM should penalize it via the Conciseness criterion.

In [ ]:
from src.data.rubric_preferences import PROBE_PROMPTS, PROBE_RESPONSES, FILLER_PARAGRAPH
from src.training.rubric_reward import evaluate_length_bias

bt_bias     = evaluate_length_bias(bt_model,     tokenizer, PROBE_PROMPTS, PROBE_RESPONSES, device)
rubric_bias = evaluate_length_bias(rubric_model, tokenizer, PROBE_PROMPTS, PROBE_RESPONSES, device)

print(f"Bradley-Terry length bias:  {bt_bias:+.3f}")
print(f"Rubric RM     length bias:  {rubric_bias:+.3f}")
print()
if bt_bias > 0 and rubric_bias < bt_bias:
    ratio = bt_bias / max(rubric_bias, 1e-6)
    print(f"Rubric RM is {ratio:.1f}× less susceptible to length exploitation.")
print()
print("Filler appended to each response:")
print(FILLER_PARAGRAPH[:200] + "...")

## 6. Results Summary

In [ ]:
print("=" * 68)
print("  RUBRIC RM vs BRADLEY-TERRY RM: RESULTS")
print("=" * 68)
print()
print(f"  {'Metric':<32} {'BT RM':>12} {'Rubric RM':>12} {'Winner':>10}")
print("  " + "-" * 70)
print(f"  {'Pairwise accuracy':<32} {bt_correct/n:>12.1%} {rubric_correct/n:>12.1%} "
      f"{'BT' if bt_correct > rubric_correct else 'Rubric':>10}")
print(f"  {'Length bias delta':<32} {bt_bias:>+12.3f} {rubric_bias:>+12.3f} "
      f"{'Rubric' if rubric_bias < bt_bias else 'BT':>10}")
print()
print("  Interpretation:")
print("  - BT RM is the right training signal for PPO/DPO (best pair ranking)")
print("  - Rubric RM is better for safety gating and OOD deployment")
print("  - Ensemble both for defence-in-depth against reward hacking")